# Data Description
- Background: Unit 1.7 官方來源證據包；官方metadata原始HTTP回應、web工具擷取原始回應JSON、manifest及歷史內包身分證據，與學生12份資料分開。
- date received: 本次查證2026-11-26；各請求UTC時間見manifest，非學生原始資料下載日期。
- Path to data file: `data/raw/unit17_supplementary_source_evidence.zip`；來源URL依包內source_manifest.json各條目。
- Unit of observation: 一份請求回應／失敗請求紀錄／身分驗證，非統計樣本。
- Sample period: 各頁面擷取時點；metadata日期不等於資料觀測日期。
- Known issues: 部分官方網域TLS失敗，未繞過驗證；web擷取JSON不是HTTP原始bytes。兩歷史說明候選URL意外回傳ZIP，已保留證據並停止重取；內包hash與學生原檔一致，不替換原檔。其他來源僅metadata對應，不宣稱最新版byte認證。
- Definition for each variable: manifest記錄URL、擷取時間、content type、成功／錯誤、檔名、bytes、SHA-256；歷史identity記錄內包member、local path及兩份hash。下方顯示原始鍵名及內容。
- Data path: `data/raw/unit17_supplementary_source_evidence.zip`。
- Role: raw（來源驗證證據，不是正式分析資料）。
- Verification status: 以下CRC、成員hash及manifest交叉核對輸出為準。
- SHA-256: `f5394c215f0db5a0124ea831d0552fe9aa26ea1c5901797a5a8fcce042f6b3bf`。
- Next processing notebook: [Unit 1.7來源與事故驗證](../code/taipei_supplementary_accident_boundary_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

In [2]:
evidence=ctx.file(ctx.config['evidence']['path'])
assert fingerprint(evidence)==ctx.config['evidence']['sha256']
z=zipfile.ZipFile(evidence)
assert z.testzip() is None
manifest=json.loads(z.read('source_manifest.json'))

## Summary

In [3]:
members=pd.DataFrame([dict(member=i.filename,bytes=i.file_size,sha256=hashlib.sha256(z.read(i.filename)).hexdigest(),zip_time=str(i.date_time)) for i in z.infolist()])
show('完整成員與hash',members)
show('歷史包身分',pd.DataFrame(json.loads(z.read('historical_source_identity.json'))))

**完整成員與hash**

,member,bytes,sha256,zip_time
0,accident_points.html,97207,6fe5d1548197f66ad495912c6b37d7e203e2d401f719cd86f8bfc1ec0e0d2e78,"(2026, 11, 26, 17, 52, 2)"
1,signals.html,83544,41a2a627631ade4f98bfc56d357270ea16d274ef84624b6770130e4242f0fe0f,"(2026, 11, 26, 17, 52, 4)"
2,crossings.html,83759,d0e4d1c40d4282486a8b9e394d2591a3a38695ab41173415558dc71dda7405e9,"(2026, 11, 26, 17, 52, 2)"
3,road_width.html,83525,8549daf165960a13865f5fb7d2eb3028b7575e90dfe4f2caae9f52bd97d0d5f5,"(2026, 11, 26, 17, 52, 4)"
4,sidewalk.html,85853,57ab9a77b74f577638164982ec3756361a7e2387cfaae42064916ecfd2614122,"(2026, 11, 26, 17, 52, 4)"
5,mrt.html,83939,5af3c801f2bce3ae76e7361b393b2a858d113137bd63f1046e36b8c9547c68ec,"(2026, 11, 26, 17, 52, 4)"
6,osm_copyright.html,21314,98f1ae74bbd894a405508c52e75104c9eced31575440041be56e7ac3dfe1014e,"(2026, 11, 26, 17, 52, 4)"
7,github_file_limits.html,213608,6bd7c4b1a0ac0a926b6e62f74b94ee3670a2ddfbf8c2368e0dfcd4e2333fe54a,"(2026, 11, 26, 17, 52, 4)"
8,history_113_landing.html,22238163,3e9aaab531d77f632e87d78ed832645902c8189aead1f226eb27ce32261c1ab6,"(2026, 11, 26, 17, 52, 30)"
9,history_114_landing.html,22171681,3de4ac9b88e57754fd1d5b28ab6317486f87b9bc1cff38580ce01708a901d236,"(2026, 11, 26, 17, 52, 30)"


**歷史包身分**

,year_label,local_path,official_wrapper,actual_media,inner_member,inner_sha256,local_sha256,byte_identical_inner,acquisition_note
0,113,data/raw/113村里界.zip,history_113_landing.html,"ZIP wrapper, not HTML",202609301102412236.zip,32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94,32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94,True,候選說明URL意外直接回傳ZIP；保留原始回應、不再重取、不替換學生檔案
1,114,data/raw/114村里界.zip,history_114_landing.html,"ZIP wrapper, not HTML",202609301104109864.zip,08d2feeced98473215703defdbe9da8f8d551f2fb221029557b813e1a3dd8845,08d2feeced98473215703defdbe9da8f8d551f2fb221029557b813e1a3dd8845,True,候選說明URL意外直接回傳ZIP；保留原始回應、不再重取、不替換學生檔案


### Sample

In [4]:
display(manifest[:10] if isinstance(manifest,list) else manifest)

[{'member': 'historical_boundaries.html',
  'url': 'https://data.gov.tw/dataset/130549',
  'status': 'failed',
  'error': '<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>'},
 {'member': 'national_2024.html',
  'url': 'https://data.gov.tw/dataset/172969',
  'status': 'failed',
  'error': '<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>'},
 {'member': 'national_2025.html',
  'url': 'https://data.gov.tw/dataset/177136',
  'status': 'failed',
  'error': '<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>'},
 {'member': 'accident_points.html',
  'url': 'https://data.taipei/dataset/detail?id=0554bac7-cbc2-4ef3-a55e-0aad3dd4ee1d',
  'final_url': 'https://data.taipei/dataset/detail?id=0554bac7-cbc2-4ef3-a55e-0aad3dd4ee1d',
  'retrieved_at_utc': '2026-11-26T09:52:03.606227+00:00',
  'bytes': 97207,
  's

### Summary Stats

In [5]:
print('CRC全部通過，證據容器bytes',evidence.stat().st_size,'SHA-256',fingerprint(evidence))
print('manifest全內容：')
display(manifest)
print('所有HTML／JSON以外的大檔成員已明記實際ZIP media；非當成HTML解碼。')
z.close()

CRC全部通過，證據容器bytes 44635308 SHA-256 f5394c215f0db5a0124ea831d0552fe9aa26ea1c5901797a5a8fcce042f6b3bf
manifest全內容：


[{'member': 'historical_boundaries.html',
  'url': 'https://data.gov.tw/dataset/130549',
  'status': 'failed',
  'error': '<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>'},
 {'member': 'national_2024.html',
  'url': 'https://data.gov.tw/dataset/172969',
  'status': 'failed',
  'error': '<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>'},
 {'member': 'national_2025.html',
  'url': 'https://data.gov.tw/dataset/177136',
  'status': 'failed',
  'error': '<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>'},
 {'member': 'accident_points.html',
  'url': 'https://data.taipei/dataset/detail?id=0554bac7-cbc2-4ef3-a55e-0aad3dd4ee1d',
  'final_url': 'https://data.taipei/dataset/detail?id=0554bac7-cbc2-4ef3-a55e-0aad3dd4ee1d',
  'retrieved_at_utc': '2026-11-26T09:52:03.606227+00:00',
  'bytes': 97207,
  's

所有HTML／JSON以外的大檔成員已明記實際ZIP media；非當成HTML解碼。
